# Train BGRL / DGI / MVGRL on **timme**

clusters=2 (R/D party labels present → NMI reported)

Shared models live in `gclib.py`; the loader lives in `train_timme.py`. This notebook just wires them together so you can iterate on the A100.

In [ ]:
import gclib, common, importlib
import train_timme as loader
importlib.reload(gclib); importlib.reload(common); importlib.reload(loader)
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
# --- load graph (symmetrized edges + ideology features) ---
# limit=... reads only the first N rows/lines for a quick sanity pass; set None for full run
data, labels = loader.load_data(limit=None)
print(data)

In [ ]:
# --- config: fixed encoder/dim across all three models for a fair comparison ---
cfg = gclib.Cfg(dim=256, layers=2, epochs=500)
cfg.clusters = 2
cfg

In [ ]:
# MVGRL diffusion view is CPU preprocessing — computed once here, reused below.
# For big graphs cache it: torch.save((data.diff_edge_index, data.diff_edge_weight), 'diff.pt')
results = gclib.run_all(data, cfg, labels=labels, models=['bgrl','dgi','mvgrl'])

In [ ]:
summary = gclib.save(results, './out/timme', 'timme')
summary